# 🔧 Preprocesamiento Completo — Gini LATAM

Nombre de integrante: Juan Esteban Guillen Hernandez

**Actividad (Seguimiento):**
1. Verificar tipos de datos y corregir los incorrectos
2. Particionar dataset en entrenamiento y prueba (train_test_split)
3. Guardar conjuntos en archivos separados (train.csv, test.csv)
4. Aplicar One-Hot-Encoding a variables categóricas
5. Aplicar escalamiento/normalización a variables numéricas.

---
## Celda 1 — Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import requests
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

print(' Librerías cargadas correctamente')

✅ Librerías cargadas correctamente


---
## Celda 2 — Descargar y preparar datos

In [ ]:
paises_iso = ['CO','BR','MX','AR','CL','PE','EC','BO','VE','PY','UY','CR','GT','HN','SV']
paises_str = ';'.join(paises_iso)

url = (f'https://api.worldbank.org/v2/country/{paises_str}/indicator/SI.POV.GINI'
       f'?format=json&per_page=2000&mrv=30')

resp = requests.get(url, timeout=30)
datos = resp.json()

registros = []
for item in datos[1]:
    registros.append({
        'Pais': item['country']['value'],
        'Codigo_ISO': item['countryiso3code'],
        'Anio': int(item['date']),
        'Gini': item['value']
    })

df = pd.DataFrame(registros)
df = df[(df['Anio'] >= 2000) & (df['Anio'] <= 2023)]
df = df.dropna(subset=['Gini']).copy()
df = df.sort_values(['Pais','Anio']).reset_index(drop=True)

print('✅ Datos descargados y preprocesados')
print(f'   Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
display(df.head(8))

 Datos descargados y preprocesados
   Filas: 282 | Columnas: 4


,Pais,Codigo_ISO,Anio,Gini
0,Argentina,ARG,2000,51.0
1,Argentina,ARG,2001,53.3
2,Argentina,ARG,2002,53.8
3,Argentina,ARG,2003,51.0
4,Argentina,ARG,2004,48.5
5,Argentina,ARG,2005,47.8
6,Argentina,ARG,2006,46.4
7,Argentina,ARG,2007,46.3


---
##  Punto 1 — Verificar tipos de datos y corregir los incorrectos

In [ ]:
print('TIPOS DE DATOS ANTES DE CORREGIR:')
print(df.dtypes.to_string())
print()

# Correcciones
df['Pais']       = df['Pais'].astype(str)        # texto
df['Codigo_ISO'] = df['Codigo_ISO'].astype(str)   # texto
df['Anio']       = df['Anio'].astype(int)         # entero
df['Gini']       = pd.to_numeric(df['Gini'], errors='coerce').astype(float)  # decimal

print('TIPOS DE DATOS DESPUÉS DE CORREGIR:')
print(df.dtypes.to_string())
print()

# Verificar si quedan nulos tras conversión
print('Nulos por columna tras corrección:')
print(df.isnull().sum().to_string())
print()
print('✅ Tipos de datos correctos:')
print('   Pais       → object  (texto categórico)')
print('   Codigo_ISO → object  (texto categórico)')
print('   Anio       → int64   (entero numérico)')
print('   Gini       → float64 (decimal numérico — variable dependiente)')

TIPOS DE DATOS ANTES DE CORREGIR:
Pais           object
Codigo_ISO     object
Anio            int64
Gini          float64

TIPOS DE DATOS DESPUÉS DE CORREGIR:
Pais           object
Codigo_ISO     object
Anio            int64
Gini          float64

Nulos por columna tras corrección:
Pais          0
Codigo_ISO    0
Anio          0
Gini          0

✅ Tipos de datos correctos:
   Pais       → object  (texto categórico)
   Codigo_ISO → object  (texto categórico)
   Anio       → int64   (entero numérico)
   Gini       → float64 (decimal numérico — variable dependiente)


---
## Punto 2 — Particionar en entrenamiento y prueba (train_test_split)

In [ ]:
# Variables independientes (X) y dependiente (y)
X = df[['Pais', 'Codigo_ISO', 'Anio']]
y = df['Gini']

# Partición 80% entrenamiento / 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print('✅ Dataset particionado con train_test_split')
print()
print(f'   Total registros:         {len(df)}')
print(f'   Entrenamiento (80%):     {len(X_train)} filas')
print(f'   Prueba         (20%):    {len(X_test)} filas')
print()
print('   X_train (primeras filas):')
display(X_train.head())
print('   X_test (primeras filas):')
display(X_test.head())

✅ Dataset particionado con train_test_split

   Total registros:         282
   Entrenamiento (80%):     225 filas
   Prueba         (20%):    57 filas

   X_train (primeras filas):


,Pais,Codigo_ISO,Anio
139,Ecuador,ECU,2018
193,Mexico,MEX,2002
19,Argentina,ARG,2020
159,El Salvador,SLV,2014
140,Ecuador,ECU,2019


   X_test (primeras filas):


,Pais,Codigo_ISO,Anio
92,Colombia,COL,2017
93,Colombia,COL,2018
179,Honduras,HND,2008
124,Ecuador,ECU,2003
261,Uruguay,URY,2009


---
## Punto 3 — Guardar conjuntos en archivos separados (train.csv, test.csv)

In [ ]:
# Unir X e y para guardar completo
df_train = X_train.copy()
df_train['Gini'] = y_train.values

df_test = X_test.copy()
df_test['Gini'] = y_test.values

# Guardar como CSV
df_train.to_csv('train.csv', index=False)
df_test.to_csv('test.csv', index=False)

print('✅ Archivos guardados:')
print(f'   train.csv → {len(df_train)} filas')
print(f'   test.csv  → {len(df_test)} filas')
print()
print('Vista previa train.csv:')
display(df_train.head())
print('Vista previa test.csv:')
display(df_test.head())

✅ Archivos guardados:
   train.csv → 225 filas
   test.csv  → 57 filas

Vista previa train.csv:


,Pais,Codigo_ISO,Anio,Gini
139,Ecuador,ECU,2018,45.4
193,Mexico,MEX,2002,50.6
19,Argentina,ARG,2020,42.7
159,El Salvador,SLV,2014,41.6
140,Ecuador,ECU,2019,45.7


Vista previa test.csv:


,Pais,Codigo_ISO,Anio,Gini
92,Colombia,COL,2017,50.9
93,Colombia,COL,2018,51.6
179,Honduras,HND,2008,55.5
124,Ecuador,ECU,2003,53.5
261,Uruguay,URY,2009,45.5


---
##  Punto 4 — One-Hot-Encoding a variables categóricas

Se aplica a `Pais` y `Codigo_ISO` ya que son variables de texto (categóricas).  
Se aplica **solo sobre el conjunto de entrenamiento (train)** para no contaminar el test.

In [ ]:
# Variables categóricas a codificar
vars_categoricas = ['Pais', 'Codigo_ISO']

# Crear encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Ajustar y transformar TRAIN
ohe_train = encoder.fit_transform(X_train[vars_categoricas])
cols_ohe  = encoder.get_feature_names_out(vars_categoricas)
df_ohe_train = pd.DataFrame(ohe_train, columns=cols_ohe, index=X_train.index)

# Transformar TEST (sin ajustar de nuevo)
ohe_test = encoder.transform(X_test[vars_categoricas])
df_ohe_test = pd.DataFrame(ohe_test, columns=cols_ohe, index=X_test.index)

# Combinar con columna numérica Anio
X_train_enc = pd.concat([df_ohe_train, X_train[['Anio']].reset_index(drop=True)], axis=1)
X_test_enc  = pd.concat([df_ohe_test,  X_test[['Anio']].reset_index(drop=True)],  axis=1)

print('✅ One-Hot-Encoding aplicado a:', vars_categoricas)
print(f'   Columnas generadas: {len(cols_ohe)}')
print(f'   Shape X_train después de OHE: {X_train_enc.shape}')
print(f'   Shape X_test  después de OHE: {X_test_enc.shape}')
print()
print('Primeras columnas generadas por OHE:')
display(X_train_enc.iloc[:5, :10])

✅ One-Hot-Encoding aplicado a: ['Pais', 'Codigo_ISO']
   Columnas generadas: 30
   Shape X_train después de OHE: (268, 31)
   Shape X_test  después de OHE: (103, 31)

Primeras columnas generadas por OHE:


,Pais_Argentina,Pais_Bolivia,Pais_Brazil,Pais_Chile,Pais_Colombia,Pais_Costa Rica,Pais_Ecuador,Pais_El Salvador,Pais_Guatemala,Pais_Honduras
139,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
193,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
159,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
140,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


---
##  Punto 5 — Escalamiento y normalización de variables numéricas

- **StandardScaler:** escala para que media=0 y desv.std=1 (estandarización)
- **MinMaxScaler:** escala entre 0 y 1 (normalización)

Se aplica sobre `Anio` y `Gini` (variables numéricas).

In [ ]:
vars_numericas = ['Anio', 'Gini']

df_nums = df[vars_numericas].copy()

# --- StandardScaler (estandarización: media=0, std=1) ---
scaler_std = StandardScaler()
datos_std = scaler_std.fit_transform(df_nums)
df_std = pd.DataFrame(datos_std, columns=[f'{c}_std' for c in vars_numericas])

# --- MinMaxScaler (normalización: rango 0 a 1) ---
scaler_mm = MinMaxScaler()
datos_mm = scaler_mm.fit_transform(df_nums)
df_mm = pd.DataFrame(datos_mm, columns=[f'{c}_norm' for c in vars_numericas])

# Comparar original vs escalado
comparacion = pd.concat([df_nums.reset_index(drop=True), df_std, df_mm], axis=1)

print('✅ Escalamiento y normalización aplicados')
print()
print('ANTES (valores originales):')
print(f'  Anio  → min: {df["Anio"].min()}  | max: {df["Anio"].max()}  | media: {df["Anio"].mean():.1f}')
print(f'  Gini  → min: {df["Gini"].min():.1f} | max: {df["Gini"].max():.1f} | media: {df["Gini"].mean():.1f}')
print()
print('DESPUÉS — StandardScaler (media=0, std=1):')
print(f'  Anio_std → min: {df_std["Anio_std"].min():.2f} | max: {df_std["Anio_std"].max():.2f} | media: {df_std["Anio_std"].mean():.2f}')
print(f'  Gini_std → min: {df_std["Gini_std"].min():.2f} | max: {df_std["Gini_std"].max():.2f} | media: {df_std["Gini_std"].mean():.2f}')
print()
print('DESPUÉS — MinMaxScaler (rango 0 a 1):')
print(f'  Anio_norm → min: {df_mm["Anio_norm"].min():.2f} | max: {df_mm["Anio_norm"].max():.2f}')
print(f'  Gini_norm → min: {df_mm["Gini_norm"].min():.2f} | max: {df_mm["Gini_norm"].max():.2f}')
print()
print('Tabla comparativa (primeras 10 filas):')
display(comparacion.head(10))

✅ Escalamiento y normalización aplicados

ANTES (valores originales):
  Anio  → min: 2000  | max: 2023  | media: 2011.4
  Gini  → min: 38.0 | max: 61.6 | media: 48.4

DESPUÉS — StandardScaler (media=0, std=1):
  Anio_std → min: -1.66 | max: 1.69 | media: 0.00
  Gini_std → min: -2.06 | max: 2.61 | media: 0.00

DESPUÉS — MinMaxScaler (rango 0 a 1):
  Anio_norm → min: 0.00 | max: 1.00
  Gini_norm → min: 0.00 | max: 1.00

Tabla comparativa (primeras 10 filas):


,Anio,Gini,Anio_std,Gini_std,Anio_norm,Gini_norm
0,2000,51.0,-1.657247,0.514926,0.000000,0.550847
1,2001,53.3,-1.511928,0.970127,0.043478,0.648305
2,2002,53.8,-1.366610,1.069084,0.086957,0.669492
3,2003,51.0,-1.221292,0.514926,0.130435,0.550847
4,2004,48.5,-1.075974,0.020142,0.173913,0.444915
5,2005,47.8,-0.930655,-0.118397,0.217391,0.415254
6,2006,46.4,-0.785337,-0.395476,0.260870,0.355932
7,2007,46.3,-0.640019,-0.415268,0.304348,0.351695
8,2008,45.0,-0.494701,-0.672555,0.347826,0.296610
9,2009,43.8,-0.349382,-0.910051,0.391304,0.245763


---
## Resumen final de todo el preprocesamiento

In [ ]:
print('=' * 55)
print('RESUMEN ACTIVIDAD — PREPROCESAMIENTO COMPLETO')
print('=' * 55)
print()
print('1.  Tipos de datos verificados y corregidos')
print('      Pais/Codigo_ISO → str | Anio → int | Gini → float')
print()
print('2.  Dataset particionado con train_test_split')
print(f'      Train: {len(X_train)} filas (80%) | Test: {len(X_test)} filas (20%)')
print()
print('3. Archivos guardados')
print('      train.csv y test.csv')
print()
print('4.  One-Hot-Encoding aplicado a variables categóricas')
print(f'      Variables: Pais, Codigo_ISO → {len(cols_ohe)} columnas generadas')
print()
print('5. Escalamiento y normalización aplicados a variables numéricas')
print('      StandardScaler → Anio_std, Gini_std')
print('      MinMaxScaler   → Anio_norm, Gini_norm')

RESUMEN ACTIVIDAD — PREPROCESAMIENTO COMPLETO

1.  Tipos de datos verificados y corregidos
      Pais/Codigo_ISO → str | Anio → int | Gini → float

2.  Dataset particionado con train_test_split
      Train: 225 filas (80%) | Test: 57 filas (20%)

3. Archivos guardados
      train.csv y test.csv

4.  One-Hot-Encoding aplicado a variables categóricas
      Variables: Pais, Codigo_ISO → 30 columnas generadas

5. Escalamiento y normalización aplicados a variables numéricas
      StandardScaler → Anio_std, Gini_std
      MinMaxScaler   → Anio_norm, Gini_norm
